# Chapter 1 — Descriptive Statistics: Network Traffic Analysis

## 📋 Latihan Soal

Lakukan analisis statistik deskriptif pada data traffic jaringan untuk memahami pola, anomali, dan karakteristik traffic normal vs malicious.

### Dataset
- 5000 records network traffic (timestamp per menit)
- 12 fitur numerik: packet_rate, byte_rate, HDP_mean, HDP_variance, dHDP_mean, dHDP_variance, entropy, unique_src_ips, unique_dst_ports, syn_ratio, icmp_ratio
- Label: `Benign` (85%) dan `Malicious` (15%)
- 50 missing values pada HDP_mean, dHDP_mean, entropy (mensimulasikan data real-world)

In [ ]:
# ============================================
# SETUP: Generate Dummy Network Traffic Dataset
# ============================================
import numpy as np
import pandas as pd

np.random.seed(42)
n_samples = 5000

# Generate realistic network traffic features
# HDP = Host-based Deviation Profile
# dHDP = delta HDP (change over time)
data = {
    'timestamp': pd.date_range('2024-01-01', periods=n_samples, freq='1min'),
    'packet_rate': np.random.exponential(scale=150, size=n_samples),
    'byte_rate': np.random.exponential(scale=50000, size=n_samples),
    'HDP_mean': np.random.normal(0.5, 0.15, n_samples),
    'HDP_variance': np.random.exponential(scale=0.02, size=n_samples),
    'dHDP_mean': np.random.normal(0.01, 0.05, n_samples),
    'dHDP_variance': np.random.exponential(scale=0.005, size=n_samples),
    'entropy': np.random.beta(2, 5, n_samples),
    'unique_src_ips': np.random.poisson(50, n_samples),
    'unique_dst_ports': np.random.poisson(30, n_samples),
    'syn_ratio': np.random.beta(1, 9, n_samples),
    'icmp_ratio': np.random.beta(1, 20, n_samples),
}

df = pd.DataFrame(data)

# Inject malicious traffic patterns (15% of data)
malicious_idx = np.random.choice(n_samples, size=750, replace=False)
df.loc[malicious_idx, 'packet_rate'] = np.random.exponential(scale=500, size=750)
df.loc[malicious_idx, 'byte_rate'] = np.random.exponential(scale=200000, size=750)
df.loc[malicious_idx, 'HDP_mean'] = np.random.normal(0.85, 0.1, 750)
df.loc[malicious_idx, 'HDP_variance'] = np.random.exponential(scale=0.08, size=750)
df.loc[malicious_idx, 'entropy'] = np.random.beta(5, 2, 750)
df.loc[malicious_idx, 'unique_src_ips'] = np.random.poisson(200, 750)
df.loc[malicious_idx, 'syn_ratio'] = np.random.beta(5, 3, 750)

df['traffic_type'] = 'Benign'
df.loc[malicious_idx, 'traffic_type'] = 'Malicious'

# Add some missing values (like real-world data)
for col in ['HDP_mean', 'dHDP_mean', 'entropy']:
    mask = np.random.choice(n_samples, size=50, replace=False)
    df.loc[mask, col] = np.nan

print(f"Dataset shape: {df.shape}")
print(f"Traffic types:\n{df['traffic_type'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

## 📖 Glosarium Feature Dataset

Dataset ini adalah **data sintetis** yang di-generate untuk mensimulasikan network traffic realistis dengan 15% malicious traffic. Berikut definisi setiap feature:

| Feature | Tipe | Definisi | Rentang Normal | Mengapa Penting |
|---------|------|----------|----------------|-----------------|
| `timestamp` | datetime | Waktu capture data (interval 1 menit) | 2024-01-01 + 5000 min | Basis analisis time-series |
| `packet_rate` | float | Jumlah paket per detik | ~150 avg (exp) | Traffic tinggi → mungkin DDoS/scan |
| `byte_rate` | float | Jumlah byte per detik | ~50K avg (exp) | Volume data abnormal → exfiltration |
| `HDP_mean` | float | **Host-based Deviation Profile** (mean) — seberapa jauh perilaku host dari baseline normal. 0 = normal, 1 = sangat menyimpang | 0.35–0.65 (μ=0.5, σ=0.15) | Host yang menyimpang = indikator compromise |
| `HDP_variance` | float | **HDP Variance** — seberapa tidak konsisten deviasi host dari waktu ke waktu | ~0.02 avg (exp) | Variance tinggi → perilaku tidak stabil (malware) |
| `dHDP_mean` | float | **Delta HDP** (mean) — perubahan/deviasi HDP antar waktu. Mengukur **laju perubahan** perilaku host | -0.04–0.06 (μ=0.01, σ=0.05) | Perubahan cepat → host sedang ter-compromise |
| `dHDP_variance` | float | **Delta HDP Variance** — konsistensi perubahan HDP | ~0.005 avg (exp) | Fluktuasi cepat → aktivitas tidak teratur |
| `entropy` | float | **Shannon Entropy** distribusi IP/port tujuan. Mengukur **keacakan** traffic. Entropy tinggi = banyak tujuan berbeda | 0.1–0.5 (β(2,5)) | Entropy tinggi → scanning, worm spreading |
| `unique_src_ips` | int | Jumlah **IP sumber unik** yang terhubung | ~50 avg (Poisson) | Banyak IP → botnet atau DDoS |
| `unique_dst_ports` | int | Jumlah **port tujuan unik** yang diakses | ~30 avg (Poisson) | Banyak port → port scanning |
| `syn_ratio` | float | Rasio paket **SYN** terhadap total paket TCP. SYN tanpa ACK = SYN flood | 0.05–0.15 (β(1,9)) | Rasio tinggi → SYN flood attack |
| `icmp_ratio` | float | Rasio paket **ICMP** terhadap total paket | 0.02–0.08 (β(1,20)) | ICMP tinggi → ping sweep / ICMP tunnel |
| `traffic_type` | str | Label kelas: **`Benign`** (85%) atau **`Malicious`** (15%) | - | Ground truth untuk supervised learning |

### 🔑 Feature Utama yang Sering Dipakai

**HDP (Host-based Deviation Profile):**
- Mengukur seberapa "aneh" perilaku sebuah host dibandingkan baseline normal
- `HDP_mean` = rata-rata deviasi (nilai lebih tinggi = lebih menyimpang)
- `HDP_variance` = seberapa tidak konsisten deviasinya
- Pada malicious traffic: HDP_mean ~0.85 (jauh dari baseline 0.5)

**dHDP (Delta HDP):**
- Mengukur **perubahan** HDP dari waktu ke waktu (turunan/deviasi pertama)
- `dHDP_mean` = rata-rata perubahan (positif = semakin menyimpang)
- `dHDP_variance` = seberapa fluktuatif perubahannya
- Berguna untuk mendeteksi host yang **sedang aktif** di-compromise (bukan yang sudah lama)

### 📊 Distribusi per Traffic Type
- **Benign:** 4250 samples (85%) — traffic normal
- **Malicious:** 750 samples (15%) — traffic dengan pattern serangan
- Missing values: 50 per kolom di `HDP_mean`, `dHDP_mean`, `entropy` (mensimulasikan data real-world)

---
## Soal 1: Mean — Rata-rata Traffic Metrics

Hitung rata-rata dari setiap fitur numerik. Bandingkan mean antara traffic Benign vs Malicious. Fitur mana yang paling besar perbedaannya?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df.groupby('traffic_type').mean()

# Jawaban:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
mean_by_type = df.groupby('traffic_type')[numeric_cols].mean()
print("=== Mean per Traffic Type ===")
print(mean_by_type.round(4))

# Hitung perbedaan
diff = mean_by_type.loc['Malicious'] - mean_by_type.loc['Benign']
diff_pct = (diff / mean_by_type.loc['Benign'] * 100).sort_values(ascending=False)
print("\n=== % Difference (Malicious vs Benign) ===")
print(diff_pct.round(2))
print(f"\n\n📌 Fitur dengan perbedaan terbesar: {diff_pct.index[0]} ({diff_pct.iloc[0]:.1f}%)")

---
## Soal 2: Median — Nilai Tengah Traffic Volume

Hitung median untuk `packet_rate` dan `byte_rate`. Apakah median jauh berbeda dari mean? Apa implikasinya terhadap distribusi data?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df['column'].median() dan bandingkan dengan df['column'].mean()

# Jawaban:
for col in ['packet_rate', 'byte_rate']:
    mean_val = df[col].mean()
    median_val = df[col].median()
    print(f"{col}:")
    print(f"  Mean:   {mean_val:.2f}")
    print(f"  Median: {median_val:.2f}")
    print(f"  Selisih: {abs(mean_val - median_val):.2f}")
    print(f"  Mean > Median → {'Ya (skew kanan)' if mean_val > median_val else 'Tidak'}")
    print()

---
## Soal 3: Mode — Kategori Traffic Paling Umum

Apa traffic type yang paling sering muncul (mode)? Berapa frekuensinya?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df['traffic_type'].mode() dan df['traffic_type'].value_counts()

# Jawaban:
mode_type = df['traffic_type'].mode()[0]
counts = df['traffic_type'].value_counts()
print(f"Mode traffic_type: {mode_type}")
print(f"\nFrekuensi:")
for t, c in counts.items():
    pct = c / len(df) * 100
    print(f"  {t}: {c} ({pct:.1f}%)")

---
## Soal 4: Variance & Standard Deviation

Hitung variance dan standard deviation dari `HDP_mean`, `entropy`, dan `syn_ratio`. Fitur mana yang paling stabil (std terkecil)?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df['col'].var() dan df['col'].std()

# Jawaban:
cols = ['HDP_mean', 'entropy', 'syn_ratio']
stats = pd.DataFrame({
    'Variance': df[cols].var(),
    'Std Dev': df[cols].std(),
    'CV (%)': df[cols].std() / df[cols].mean() * 100  # Coefficient of Variation
})
print(stats.round(4))
print(f"\n📌 Fitur paling stabil (std terkecil): {stats['Std Dev'].idxmin()}")
print(f"📌 Fitur paling variatif (CV tertinggi): {stats['CV (%)'].idxmax()}")

---
## Soal 5: Range — Rentang Nilai Minimum & Maximum

Tentukan range (min dan max) dari setiap fitur numerik. Apakah ada nilai yang tidak masuk akal (misal: negatif untuk metric yang harusnya positif)?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: df[numeric_cols].min() dan df[numeric_cols].max()

# Jawaban:
range_df = pd.DataFrame({
    'Min': df[numeric_cols].min(),
    'Max': df[numeric_cols].max(),
    'Range': df[numeric_cols].max() - df[numeric_cols].min()
})
print(range_df.round(4))

# Cek nilai negatif yang tidak seharusnya
print("\n⚠️ Cek nilai negatif:")
for col in ['packet_rate', 'byte_rate', 'HDP_variance', 'dHDP_variance', 'entropy',
            'unique_src_ips', 'unique_dst_ports', 'syn_ratio', 'icmp_ratio']:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"  {col}: {neg_count} nilai negatif")
    else:
        print(f"  {col}: OK (tidak ada nilai negatif)")

---
## Soal 6: Percentiles — Threshold Anomali

Hitung percentil 25, 50, 75, dan 95 dari `packet_rate` dan `HDP_mean`. Gunakan percentil 95 sebagai threshold untuk mendeteksi potensi anomali. Berapa banyak data yang melebihi threshold ini?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: np.percentile() atau df.quantile()

# Jawaban:
percentiles = [25, 50, 75, 95]
for col in ['packet_rate', 'HDP_mean']:
    print(f"\n{col}:")
    for p in percentiles:
        val = df[col].quantile(p / 100)
        print(f"  P{p}: {val:.4f}")
    
    threshold_95 = df[col].quantile(0.95)
    above = (df[col] > threshold_95).sum()
    malicious_above = df.loc[df[col] > threshold_95, 'traffic_type'].value_counts()
    print(f"  Data di atas P95: {above} ({above/len(df)*100:.1f}%)")
    print(f"  Komposisi: {dict(malicious_above)}")

---
## Soal 7: Quartiles & IQR — Deteksi Outlier

Hitung Q1, Q2, Q3, dan IQR untuk `packet_rate`. Gunakan rumus outlier: nilai < Q1 - 1.5×IQR atau > Q3 + 1.5×IQR. Berapa banyak outlier? Apakah outlier cenderung malicious?

In [ ]:
# TULIS JAWABAN DI SINI
# Petunjuk: Q1 = df.quantile(0.25), Q3 = df.quantile(0.75), IQR = Q3 - Q1

# Jawaban:
col = 'packet_rate'
Q1 = df[col].quantile(0.25)
Q2 = df[col].quantile(0.50)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"{col}:")
print(f"  Q1: {Q1:.2f}")
print(f"  Q2 (Median): {Q2:.2f}")
print(f"  Q3: {Q3:.2f}")
print(f"  IQR: {IQR:.2f}")
print(f"  Batas bawah: {lower_bound:.2f}")
print(f"  Batas atas: {upper_bound:.2f}")

outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
print(f"\nTotal outlier: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
print(f"\nKomposisi outlier:")
print(outliers['traffic_type'].value_counts())

mal_rate = (outliers['traffic_type'] == 'Malicious').sum() / len(outliers) * 100
print(f"\n📌 {mal_rate:.1f}% outlier adalah traffic Malicious")
print(f"📌 IQR method berhasil mendeteksi {(outliers['traffic_type'] == 'Malicious').sum()} malicious traffic dari total {(df['traffic_type'] == 'Malicious').sum()}")

---
## 📊 Ringkasan Chapter 1

Dari analisis statistik deskriptif network traffic:
- **Mean & Median** menunjukkan distribusi skew kanan pada packet_rate dan byte_rate
- **Mode** menunjukkan traffic Benign dominan (85%)
- **Std Dev & CV** mengidentifikasi fitur paling variatif
- **Percentil 95** bisa digunakan sebagai baseline threshold anomali
- **IQR method** efektif mendeteksi outlier yang cenderung malicious